In [36]:
import pandas as pd
import polars as pl
# Loading to db
# importing os module for environment variables
import os
# importing necessary functions from dotenv library
from dotenv import load_dotenv 
# loading variables from .env file
load_dotenv() 
import psycopg2
from sqlalchemy import create_engine

## Load in cleaned_raw_pubmed_data as a LAZYFRAME

In [37]:
final_df = pl.scan_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/Data Files/cleaned_raw_pubmed_data.csv")

In [38]:
final_df.select(pl.len()).collect().item()
# 10,206,788

10206788

## Drop abstract to reduct CSV size

In [5]:
final_df_drop = final_df.drop(['abstract'])

In [7]:
final_df_drop.columns

/var/folders/tk/qn93wpd15j534g_2qn8kchdh0000gn/T/ipykernel_2641/2748717140.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  final_df_drop.columns


['title', 'journal', 'date', 'authors', 'doi']

## Store no abstract LAZYFRAME as a CSV

In [8]:
final_df_drop.sink_csv('dropped_raw_pubmed_data.csv')

## Load in new CSV

In [9]:
test_df = pd.read_csv("/Users/shantibrodnick/Downloads/510Capstone/GenderDisparityinResearch/notebooks/Article_Data_APIs/dropped_raw_pubmed_data.csv")

In [10]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10206788 entries, 0 to 10206787
Data columns (total 5 columns):
 #   Column   Dtype
---  ------   -----
 0   title    str  
 1   journal  str  
 2   date     str  
 3   authors  str  
 4   doi      str  
dtypes: str(5)
memory usage: 3.0 GB


## Check min and max dates

In [12]:
print(test_df['date'].max())
print(test_df['date'].min())

['Aileen M.Gariepy', 'Daniel J.Tancredi', 'CarrieLewis', 'DianaZuckerman', 'Eleanor BimlaSchwarz']
1781-06-01


In [ ]:
# Showing the row with the maximum value in a column
test_df.idxmax()

title      9234900
journal     517681
date       9133407
authors        219
doi        3675590
dtype: int64

In [ ]:
pd.set_option('display.max_colwidth', None)
print(test_df.iloc[[9133407]]['title'])

9133407    Comparing Essure® and Tubal Ligation to Prevent Pregnancy
Name: title, dtype: str


In [18]:
test_df.isna().sum()

title            0
journal       4976
date             0
authors          0
doi        1095306
dtype: int64

## Fix weird formatted row

In [21]:
test_df[test_df['doi'].isnull()].sort_values('date')

,title,journal,date,authors,doi
877411,A Case of Epilepsy Successfully Treated.,The London medical journal,1781-06-01,[],NaN
9548108,Account of a Woman Who Had the Small Pox durin...,The London medical journal,1781-09-01,[],NaN
8105221,An Account of the Good Effects of Fixed Air in...,The London medical journal,1783-01-01,[],NaN
10145356,"A Short History of Three Cases of Women, Who W...",The London medical journal,1784-10-01,[],NaN
2612226,"A Case, Shewing the Efficacy of Flowers of Zin...",The London medical journal,1786-01-01,['JohnLind'],NaN
...,...,...,...,...,...
8299829,Maternal Vitamin D Deficiency and Its Impact o...,Mymensingh medical journal : MMJ,2026-06-30,"['JSaha', 'JAra', 'SAkter', 'S FShetu', 'SSidd...",NaN
9184565,Associated Coronary and Cardiac Anomalies amon...,Mymensingh medical journal : MMJ,2026-06-30,"['U K MAra', 'M ZHussain', 'C MAhmed', 'TIslam...",NaN
10103351,Short-term Outcome of Grisotti Flap Oncoplasti...,Mymensingh medical journal : MMJ,2026-06-30,"['K RMajumder', 'MRassell', 'DMaitra', 'SPerve...",NaN
10104533,Outcome of Primary Total Hip Arthroplasty in D...,Mymensingh medical journal : MMJ,2026-06-30,"['A N MHasnat', 'USalma', 'S PShimul', 'AHossa...",NaN


In [27]:
test_df.at[9133407, 'doi'] = test_df.at[9133407, 'authors']
# 10.25302/10.2021.CER.160936359

# authors: ['Aileen M.Gariepy', 'Daniel J.Tancredi', 'CarrieLewis', 'DianaZuckerman', 'Eleanor BimlaSchwarz']

# 2021

In [28]:
test_df.iloc[[9133407]]


,title,journal,date,authors,doi
9133407,Comparing Essure® and Tubal Ligation to Prevent Pregnancy,2021,"['Aileen M.Gariepy', 'Daniel J.Tancredi', 'CarrieLewis', 'DianaZuckerman', 'Eleanor BimlaSchwarz']",10.25302/10.2021.CER.160936359,10.25302/10.2021.CER.160936359


In [29]:
test_df.at[9133407, 'authors'] = test_df.at[9133407, 'date']

In [30]:
test_df.iloc[[9133407]]

,title,journal,date,authors,doi
9133407,Comparing Essure® and Tubal Ligation to Prevent Pregnancy,2021,"['Aileen M.Gariepy', 'Daniel J.Tancredi', 'CarrieLewis', 'DianaZuckerman', 'Eleanor BimlaSchwarz']","['Aileen M.Gariepy', 'Daniel J.Tancredi', 'CarrieLewis', 'DianaZuckerman', 'Eleanor BimlaSchwarz']",10.25302/10.2021.CER.160936359


In [31]:
test_df.at[9133407, 'date'] = test_df.at[9133407, 'journal']

In [33]:
test_df.at[9133407, 'journal'] = None

In [ ]:
test_df.at[9133407, 'authors'] = test_df.at[9133407, 'date']
test_df.iloc[[9133407]]
test_df.at[9133407, 'date'] = test_df.at[9133407, 'journal']
test_df.at[9133407, 'journal'] = None

In [34]:
test_df.iloc[[9133407]]

,title,journal,date,authors,doi
9133407,Comparing Essure® and Tubal Ligation to Prevent Pregnancy,NaN,2021,"['Aileen M.Gariepy', 'Daniel J.Tancredi', 'CarrieLewis', 'DianaZuckerman', 'Eleanor BimlaSchwarz']",10.25302/10.2021.CER.160936359


## New dataframe!

In [35]:
test_df

,title,journal,date,authors,doi
0,Anti-tumor necrosis factor-alpha antibody treatment reduces serum CXCL16 levels in patients with rheumatoid arthritis.,Rheumatology international,2006-10-20,"['YasunoriKageyama', 'EijiTorikai', 'AkiraNagano']",10.1007/s00296-006-0241-1
1,[Quantification of expression of leukotriene B4 inducing tumor necrosis factor-alpha and interleukin-1beta at mRNA level in synovial membrane cells of rheumatoid arthritis by real-time quantitative PCR].,Beijing da xue xue bao. Yi xue ban = Journal of Peking University. Health sciences,2006-10-28,"['Zhan-kunChen', 'Hou-shanLv']",NaN
2,Interleukin-7 induced immunopathology in arthritis.,Annals of the rheumatic diseases,2006-10-14,"['S A YHartgring', 'J W JBijlsma', 'F P J GLafeber', 'J A Gvan Roon']",10.1136/ard.2006.058479
3,High-throughput quantitation of metabolically labeled anionic glycoconjugates by scintillation proximity assay utilizing binding to cationic dyes.,"Methods in molecular biology (Clifton, N.J.)",2006-10-31,"['Karen JRees-Milton', 'Tassos PAnastassiades']",10.1385/1-59745-167-3:267
4,Pulsed electrical stimulation to defer TKA in patients with knee osteoarthritis.,Orthopedics,2006-10-26,"['Michael AMont', 'David SHungerford', 'Jacques RCaldwell', 'Phillip SRagland', 'Kent CHoffman', 'Y DavidHe', 'Lynne CJones', 'Thomas MZizic']",10.3928/01477447-20061001-13
...,...,...,...,...,...
10206783,Reframing precision nutrition in irritable bowel syndrome: a mechanism-informed conceptual framework for responder prediction and clinical translation.,Frontiers in immunology,2026-06-15,"['YaZhou', 'ZhenLi', 'YuzhouChu', 'ZhijiaZhou', 'TaoZhang', 'NingYi', 'WuquanSun', 'JuntaoYan', 'ZhenYan', 'AnningZhu']",10.3389/fimmu.2026.1809221
10206784,Serum neurofilament light chain in paediatric patients treated with natalizumab for highly active multiple sclerosis.,"Multiple sclerosis (Houndmills, Basingstoke, England)",2026-06-23,"['BrendaHuppke', 'Marie-ChristineReinert', 'WiebkeStark', 'JuttaGärtner', 'PeterHuppke']",10.1177/13524585261450816
10206785,Solvent-free engineering of a co-amorphous efavirenz-ritonavir system by hot-melt extrusion: Solid-state stabilisation and improved bioavailability.,International journal of pharmaceutics,2026-06-11,"['ShubhamGhatole', 'KoustavTaladhi', 'MamtaKumari', 'Sai SharanyaPulimamidi', 'Roshan MBorkar', 'SubhadeepRoy', 'SantanuKaity']",10.1016/j.ijpharm.2026.127068
10206786,Longitudinal assessment of myocardial involvement in PASC-CVS: a single-center study from China based on multiparametric CMR.,Frontiers in cardiovascular medicine,2026-06-03,"['AiShang', 'ShanYang', 'JiayeTao', 'JieShen', 'YiZhan', 'YinwenGan', 'ZhiyongZhang', 'HangJin', 'FeiShan']",10.3389/fcvm.2026.1725291


## Drop authors

In [42]:
final_pubmed_data_4 = test_df.drop(labels='authors', axis=1)

In [43]:
final_pubmed_data_4.info()

<class 'pandas.DataFrame'>
RangeIndex: 10206788 entries, 0 to 10206787
Data columns (total 4 columns):
 #   Column   Dtype
---  ------   -----
 0   title    str  
 1   journal  str  
 2   date     str  
 3   doi      str  
dtypes: str(4)
memory usage: 2.0 GB


## Save no abstract no author, fixed error dataframe as a CSV

In [46]:
final_pubmed_data_4.to_csv("final_raw_pubmed_data.csv")

## CSV to DATABASE

In [44]:
# Connect to the database
conn = psycopg2.connect(
    dbname=os.getenv("DBNAME"),
    user=os.getenv("DBUSER"),
    password=os.getenv("DBPASSWORD"),
    port=os.getenv("DBPORT"),
    host=os.getenv("DBHOST")
)
conn_string=os.getenv("CONNSTRING")

# Create engine
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBUSER')}:{os.getenv('DBPASSWORD')}"
    f"@{os.getenv('DBHOST')}:{os.getenv('DBPORT')}/{os.getenv('DBNAME')}"
)

In [45]:
final_pubmed_data_4.to_sql("raw_pubmed_data", engine, if_exists="append", index=False, chunksize=1000)

10206788